# Professional English Coach — optional media inspection

Use this Colab notebook to inspect the technical metadata of authored lesson audio and optionally preview it. It does not generate speech, call VoiceStudio, or connect to the learner-facing app. Inspection is optional and is not a publication gate.

The current sample is generated locally with FFmpeg/libflite. Its pronunciation and output rights have not been independently reviewed; the project owner authorized its use for personal/family learning and accepted the existing public Site exposure. This notebook does not verify or grant rights.

**Privacy:** Colab runs notebook code on a Google-managed runtime. Upload only authored lesson clips you choose to process there. Never upload learner voice recordings, private reference voices, credentials, or confidential material. Do not mount Google Drive for this workflow.


In [ ]:
from google.colab import files
from pathlib import Path

uploaded = files.upload()  # Select one authored lesson-media candidate.
if not uploaded:
    raise RuntimeError("Upload one approved-for-review lesson candidate first.")
filename = next(iter(uploaded))
clip_path = Path(filename)
print(f"Loaded {clip_path.name} ({clip_path.stat().st_size:,} bytes)")


In [ ]:
import json
import mimetypes
import shutil
import subprocess

ffprobe = shutil.which("ffprobe")
if not ffprobe:
    raise RuntimeError("ffprobe is unavailable in this runtime; inspect the file locally and record metadata manually.")
result = subprocess.run(
    [ffprobe, "-v", "error", "-show_entries", "format=duration,size:stream=codec_name,sample_rate,channels", "-of", "json", str(clip_path)],
    check=True, capture_output=True, text=True
)
metadata = json.loads(result.stdout)
print(json.dumps(metadata, indent=2))


In [ ]:
from IPython.display import Audio, display

display(Audio(filename=str(clip_path)))


## Optional preview

Listen to the final encoded file above if useful. The app also contains a transcript and a skip route. No human checklist or licensing decision is required to use this notebook.


In [ ]:
# Optional transcript/provenance notes for the output file below.
script = "The main trade-off is higher latency in exchange for better isolation."
source_record = "FFmpeg 6.1.1 / libflite voice kal; local synthetic voice; no reference recording"
transcript_status = "provided-in-application; not independently compared by this notebook"
rights_status = "not-independently-reviewed"


In [ ]:
from datetime import date

asset_record = {
    "file": clip_path.name,
    "contentType": mimetypes.guess_type(clip_path.name)[0],
    "inspectionDate": date.today().isoformat(),
    "technicalMetadata": metadata,
    "script": script,
    "sourceRecord": source_record,
    "transcriptStatus": transcript_status,
    "rightsStatus": rights_status,
    "ownerAuthorization": "personal-and-family-use; public-hosting-risk-accepted",
}
output_path = Path("media-inspection-result.json")
output_path.write_text(json.dumps(asset_record, indent=2) + "\n", encoding="utf-8")
print(output_path.read_text())


In [ ]:
files.download(str(output_path))
